# Topic 2 - Evaluation harness for TravelMind

**What this notebook is.** A repeatable way to answer one question: *did a change
make the agent better or worse?* For a normal program you would diff the output
against a known answer. An agent has no single right answer, so instead you score
the output against **acceptance criteria** and compare the score to a baseline.

**The shape of the harness (cheapest grader first):**

| Layer | Grader | Use it when |
|-------|--------|-------------|
| 1 | Deterministic checks (substring, shape) | Almost always. Free and instant. |
| 2 | LLM-as-judge | A substring cannot grade it: tone, completeness, no-invention. |
| 3 | RAGAS faithfulness | The answer is grounded in a knowledge base. |

**Where it fits in the pipeline.** This notebook writes `eval_report.json`. The
quality gate (Topic 4) reads that file and decides whether the build may ship.
So the number you compute here is not academic; it gates the release.

We build it up one layer at a time. Read top to bottom.

## Setup

Pick the path that matches where you are running.

**VS Code (local), three steps:**
1. Create and activate a virtual environment:
   `python -m venv .venv` then `source .venv/bin/activate` (Windows: `.venv\Scripts\activate`).
2. Give the SDK AWS credentials. Easiest for a workshop: `aws configure`
   (writes `~/.aws/credentials`). In production you would use an IAM role, not keys.
3. Install dependencies: `pip install -r requirements.txt`.

**Google Colab, three steps:**
1. `!pip install strands-agents ragas langchain-aws boto3`
2. Set credentials as environment variables (do NOT paste keys into a shared
   notebook; use Colab Secrets and read them):
   `import os; os.environ["AWS_ACCESS_KEY_ID"]=...; os.environ["AWS_SECRET_ACCESS_KEY"]=...; os.environ["AWS_DEFAULT_REGION"]="us-east-1"`
3. Upload `config.py`, `travelmind_agent.py`, and `golden_set.jsonl` to the
   session (Files panel) so the imports below resolve.

All model ids, the region, and the thresholds live in `config.py`. Change them
there, never inline.

In [ ]:
# --- imports and configuration ---
import json

# config.py is the single source of truth (model ids, region, thresholds).
import config

# The system under test. get_agent() builds the Bedrock-backed agent lazily,
# so importing the module does not by itself require AWS.
from travelmind_agent import get_agent, set_system_prompt, SYSTEM_PROMPT

# Load the golden set once. Each line is one case with its acceptance criteria.
with open("golden_set.jsonl") as f:
    CASES = [json.loads(line) for line in f if line.strip()]

print(f"loaded {len(CASES)} golden cases")
print("checks used:", sorted({c["check"] for c in CASES}))

## Layer 1 - deterministic checks

The cheapest grader. It looks at the agent's reply as lowercase text and applies
three rules taken from the case:

- `must`     : every listed phrase must appear.
- `any_of`   : at least one listed phrase must appear (use when several phrasings
               are acceptable, e.g. "could not find" or "not found").
- `must_not` : none of the listed phrases may appear (this is how we catch an
               invented flight number for a bad PNR).

Notice the check function is **pure**: it takes a reply string and a case and
returns a boolean. It does not call the agent. That separation matters because
the pure check is itself testable and deterministic, while the agent call is the
part that needs AWS.

In [ ]:
def substring_check(reply: str, case: dict) -> bool:
    """Apply must / any_of / must_not rules to a reply. Pure: no model call."""
    text = reply.lower()                       # compare case-insensitively, once

    # every 'must' phrase has to be present
    if not all(p.lower() in text for p in case.get("must", [])):
        return False

    # at least one 'any_of' phrase has to be present (only when any_of is given)
    any_of = case.get("any_of", [])
    if any_of and not any(p.lower() in text for p in any_of):
        return False

    # no 'must_not' phrase may be present (this guards against invention)
    if any(p.lower() in text for p in case.get("must_not", [])):
        return False

    return True


# A quick self-check of the checker itself, with no agent involved.
assert substring_check("Your flight is CANCELLED due to weather.",
                       {"must": ["cancelled", "weather"], "must_not": ["confirmed"]}) is True
assert substring_check("Try flight AI-318 at 18:40.",
                       {"any_of": ["ai-318", "6e-552"]}) is True
assert substring_check("Here is flight AI-318.",            # a bad PNR must not yield a flight
                       {"any_of": ["not found"], "must_not": ["ai-3"]}) is False
print("substring_check behaves correctly")

### Running one deterministic case end to end

This is the part that needs AWS: we call the agent, then grade its reply with the
pure checker. `str(result)` turns the agent's result object into its text.

In [ ]:
def run_substring_case(case: dict) -> dict:
    """Call the agent on one case and grade the reply. Needs AWS."""
    agent = get_agent()
    reply = str(agent(case["input"]))          # the model decides the wording
    passed = substring_check(reply, case)
    return {"id": case["id"], "passed": passed, "reply": reply}

# Example (uncomment when AWS is configured):
# print(run_substring_case(CASES[0]))

## Layer 2 - LLM-as-judge

Some properties cannot be checked with a substring: *did the answer actually
explain the cause AND offer an option*, *did it stay polite*, *did it avoid
inventing a policy*. For those you ask a stronger model to score the reply
against a short rubric.

Three things keep a judge honest. Skip them and the score is noise:

1. **Ground it with FACTS.** Pass the judge the ground truth from the tools.
   Without facts the judge is guessing too.
2. **Constrain the output** to JSON, so you parse a number instead of prose.
3. **Cancel position bias** for pairwise comparisons by scoring both orders,
   (A, B) and (B, A), and averaging. Judges tend to favour whichever answer comes
   first; this removes that. (Our cases are pointwise, so we note it for when you
   compare two candidates head to head.)

The judge model is **stronger** than the agent model (see `config.JUDGE_MODEL_ID`):
a grader should be at least as capable as the thing it grades.

In [ ]:
import json as _json

def judge(question: str, reply: str, facts: str, rubric_focus: str) -> dict:
    """Score a reply 1-5 against a rubric, grounded with FACTS.
    Returns {'score': int, 'reason': str}. Needs AWS."""
    from strands import Agent
    from strands.models import BedrockModel

    rubric = (
        "You are a strict QA grader. Score the REPLY from 1 to 5 on: "
        "correctness against FACTS, completeness, and no invented "
        f"PNR/flight/policy. Pay special attention to: {rubric_focus}. "
        'Return ONLY JSON: {"score": <1-5>, "reason": "<one short line>"}.'
    )
    grader = Agent(
        model=BedrockModel(model_id=config.JUDGE_MODEL_ID, region_name=config.REGION),
        system_prompt=rubric,
    )
    raw = str(grader(f"QUESTION: {question}\nFACTS: {facts}\nREPLY: {reply}"))
    try:
        return _json.loads(raw)                 # the rubric forces JSON; parse it
    except _json.JSONDecodeError:
        # Defensive: if the model wraps JSON in prose, fail closed (score 0)
        # rather than crash the whole eval run on one bad parse.
        return {"score": 0, "reason": "judge did not return valid JSON"}


def run_judge_case(case: dict, pass_at: int = 4) -> dict:
    """Run a judge case. A score at or above pass_at counts as a pass."""
    agent = get_agent()
    reply = str(agent(case["input"]))
    verdict = judge(case["input"], reply, case.get("facts", ""), case.get("rubric_focus", ""))
    return {"id": case["id"], "passed": verdict["score"] >= pass_at,
            "score": verdict["score"], "reason": verdict["reason"], "reply": reply}

# Example (uncomment when AWS is configured):
# print(run_judge_case(next(c for c in CASES if c["check"] == "judge")))

## Layer 3 - RAGAS faithfulness

Use this only when the answer is grounded in retrieved context (our policy
knowledge base from Day 4). **Faithfulness** measures the fraction of claims in
the answer that the retrieved context actually supports. It needs the retrieved
chunks, so it does not apply to the pure booking flow (there is no retrieval
there).

We use the current RAGAS **collections API**: build a `SingleTurnSample`, then
score it with a metric object. The evaluator LLM is any LangChain chat model
wrapped for RAGAS; here we back it with Bedrock.

In a Jupyter cell you can `await` directly, which is why the scoring line uses
`await`.

In [ ]:
# RAGAS scoring, defined as an async function so it is easy to await in a cell.
async def faithfulness_score(question: str, reply: str, contexts: list) -> float:
    """Return faithfulness in [0, 1]: claims supported by context / total claims.
    Needs AWS and the ragas + langchain-aws packages."""
    from ragas.dataset_schema import SingleTurnSample
    from ragas.metrics import Faithfulness
    from ragas.llms import LangchainLLMWrapper
    from langchain_aws import ChatBedrock

    # Wrap any LangChain chat model so RAGAS can call it as its evaluator.
    evaluator = LangchainLLMWrapper(
        ChatBedrock(model_id=config.JUDGE_MODEL_ID, region_name=config.REGION)
    )

    sample = SingleTurnSample(
        user_input=question,
        response=reply,
        retrieved_contexts=contexts,            # the chunks the answer was built from
    )
    return await Faithfulness(llm=evaluator).single_turn_ascore(sample)

In [ ]:
# Run the single grounded case. Top-level await is valid in Jupyter.
# (Uncomment when AWS is configured.)
#
# ragas_case = next(c for c in CASES if c["check"] == "ragas")
# agent = get_agent()
# reply = str(agent(ragas_case["input"]))
# score = await faithfulness_score(ragas_case["input"], reply, ragas_case["contexts"])
# print(f"faithfulness {score:.2f}  (pass if >= 0.80)")

## The router - one grader per case

`grade_case` dispatches each case to the right grader by its `check` field. This
keeps the run loop simple: it does not care how a case is graded, only whether it
passed. RAGAS is async, so the router handles substring and judge (both sync) and
we score the one RAGAS case separately above. Keeping the sync and async paths
apart avoids tangling an event loop into the main loop, which is the kind of
over-engineering this kit avoids.

In [ ]:
def grade_case(case: dict) -> dict:
    """Grade a sync case (substring or judge). Needs AWS. The returned dict
    always has 'id', 'passed', and 'type'."""
    if case["check"] == "substring":
        r = run_substring_case(case)
    elif case["check"] == "judge":
        r = run_judge_case(case)
    elif case["check"] == "ragas":
        # handled in the awaited cell above; mark as skipped here so the loop
        # does not silently treat it as a failure.
        r = {"id": case["id"], "passed": None, "note": "scored separately (async)"}
    else:
        raise ValueError(f"unknown check: {case['check']}")
    r["type"] = case["type"]                    # carry 'functional' / 'safety' through
    return r

## Run the whole golden set

We compute the two numbers the gate cares about:

- **eval_pass_rate**: passed / graded, across all functional and safety cases.
- **safety_pass_rate**: the same, but over the `safety` cases only. Safety is a
  separate, harder bar (the gate requires 100%), so we track it on its own.

Both ignore the async RAGAS case (`passed is None`); fold its result in by hand
after you run the awaited cell, or extend the loop to await it.

In [ ]:
def run_eval(cases):
    """Grade all sync cases and compute pass rates. Needs AWS."""
    results = [grade_case(c) for c in cases if c["check"] != "ragas"]

    graded = [r for r in results if r["passed"] is not None]
    overall = sum(r["passed"] for r in graded) / len(graded)

    safety = [r for r in graded if r["type"] == "safety"]
    safety_rate = sum(r["passed"] for r in safety) / len(safety) if safety else 1.0

    return {"pass_rate": overall, "safety_pass_rate": safety_rate, "by_case": results}

# Example (uncomment when AWS is configured):
# report = run_eval(CASES)
# print(f"overall {report['pass_rate']:.0%}   safety {report['safety_pass_rate']:.0%}")
# for r in report["by_case"]:
#     mark = "PASS" if r["passed"] else "FAIL"
#     print(f"  {r['id']:>3}  {mark}  ({r['type']})")

## Prompt regression - the catch

This is the moment the harness earns its keep. A teammate rewrites the system
prompt to be more concise. It reads better. Does it score better?

We evaluate the same golden set under two prompt versions and compare. The
decision rule is simple and strict: **promote v7 only if its pass rate is at
least v6's.** A prettier prompt that scores lower does not ship.

In [ ]:
# Two prompt versions. v6 is the current production prompt. v7 is the concise
# rewrite under review. Everything else (model, tools, cases) is held constant,
# so any difference in score is due to the prompt alone.
V6 = SYSTEM_PROMPT                              # the current, fuller prompt
V7 = (
    "You are TravelMind. Look up the PNR and answer briefly. "
    "Never invent a PNR, flight, or policy."
)                                              # concise, but drops 'explain + offer an option'


def eval_prompt(version_prompt):
    """Swap the system prompt, then run the eval. set_system_prompt invalidates
    the cached agent so the new prompt actually takes effect."""
    set_system_prompt(version_prompt)
    return run_eval(CASES)


# Example (uncomment when AWS is configured):
# r6 = eval_prompt(V6)
# r7 = eval_prompt(V7)
# print(f"v6 {r6['pass_rate']:.0%}   v7 {r7['pass_rate']:.0%}")
# regressed = [a["id"] for a, b in zip(r6["by_case"], r7["by_case"])
#              if a["passed"] and not b["passed"]]
# print("regressed under v7:", regressed)      # expect g6 (concise dropped 'offer an option')
# assert r7["pass_rate"] >= r6["pass_rate"], "do not promote: v7 regressed"
#
# set_system_prompt(V6)                         # restore the good prompt before shipping

## Emit the artifact for the gate

The gate (Topic 4) does not re-run the eval; it reads this file. Writing the
report is what connects this notebook to the pipeline.

In [ ]:
def write_eval_report(report, path="eval_report.json"):
    """Persist the numbers the gate will read."""
    out = {
        "pass_rate": round(report["pass_rate"], 4),
        "safety_pass_rate": round(report["safety_pass_rate"], 4),
        "by_case": report["by_case"],
    }
    with open(path, "w") as f:
        json.dump(out, f, indent=2)
    print(f"wrote {path}")

# Example (uncomment when AWS is configured):
# write_eval_report(run_eval(CASES))

## What changes in production

- **Bind every eval run to a prompt version.** A score with no version is not a
  comparison. Store the version next to the number.
- **Re-run the whole set on a model swap.** A new model re-rolls every behaviour;
  yesterday's pass rate does not carry over.
- **Grow the golden set from real incidents.** Every production failure becomes a
  new case, so the same bug cannot ship twice.
- **Calibrate the judge against human labels** on a sample. Trust the judge only
  as far as it agrees with people; it is triage, not ground truth.
- **Do not over-fit the prompt to the golden set.** Hold out a few cases the
  prompt author does not see, the way you keep a test set in ML.